> **Production version** — functions are imported from `src/mrta/` instead of defined inline.  
> Original teaching version: [`../tutorials/`](../tutorials/)  
> Install first: `pip install -e ".[all]"`


In [9]:
from mrta.evaluation.eval_pipeline import run_eval
from mrta.evaluation.metrics import (
    answer_relevance,
    citation_correctness,
    faithfulness,
    hallucination_rate,
)
from mrta.observability.logging import StructuredLogger

# Phase 9 — Evaluation, Observability & Docker

**Goal:** Make the project look senior-level. We add three production-shaped layers on top of the working RAG system:

1. **Evaluation** — a small benchmark + Ragas metrics for groundedness, faithfulness, and answer relevance.
2. **Observability** — structured JSONL logs with per-run latency, tokens, cited pages, and retrieval traces.
3. **Deployment** — a `docker-compose.yml` that brings up backend + frontend + Qdrant with one command.

Most candidates skip these. That is exactly why doing them is portfolio gold.


## 9.1 — Build a tiny benchmark

Five hand-labeled QA pairs over *Attention Is All You Need*. In a real project you would have 50-200; this is enough to show the pipeline.


In [10]:
import os, sys, json
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").is_file() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))

BENCH = Path("data/processed/benchmark.jsonl")
BENCH.parent.mkdir(parents=True, exist_ok=True)

benchmark = [
    {
        "question": "What is the dimensionality of the model and the feed-forward inner layer in the base Transformer?",
        "expected_pages": [3, 5],
        "expected_substrings": ["512", "2048"],
    },
    {
        "question": "How many attention heads does the base Transformer use?",
        "expected_pages": [5],
        "expected_substrings": ["8"],
    },
    {
        "question": "What positional encoding does the Transformer use and why?",
        "expected_pages": [6],
        "expected_substrings": ["sinusoid"],
    },
    {
        "question": "What BLEU score did the base Transformer achieve on WMT 2014 English-to-German?",
        "expected_pages": [8],
        "expected_substrings": ["27.3", "BLEU"],
    },
    {
        "question": "Why is scaled dot-product attention scaled by 1/sqrt(d_k)?",
        "expected_pages": [4],
        "expected_substrings": ["soft", "magnitude"],
    },
]

with BENCH.open("w") as f:
    for r in benchmark:
        f.write(json.dumps(r) + "\n")
print(f"Wrote {len(benchmark)} questions to {BENCH}")


Wrote 5 questions to data/processed/benchmark.jsonl


## 9.2 — Custom metrics (no external deps)

Start with metrics you can compute yourself. They are interpretable and need no LLM judge.


In [11]:
from mrta.core.llm import LLMClient
from mrta.core.rag_pipeline import rag_query
from mrta.evaluation.metrics import (
    answer_relevance,
    citation_correctness,
    faithfulness,
    hallucination_rate,
)
from mrta.retrieval.embedder import Embedder
from mrta.retrieval.vector_store import VectorStore

embedder = Embedder()
store = VectorStore.load("data/vector_store/aiayn", embedder)  # requires pre-built index
llm = LLMClient()
print("Components loaded.")

Components loaded.


In [12]:
print(embedder.model_name)
print(llm.model)


nomic-embed-text
llama3.2:latest


## 9.3 — Run the benchmark


In [13]:
import pandas as pd

rows = []
for q in benchmark:
    result = rag_query(q["question"], store, llm)
    rows.append({
        "question": q["question"][:60] + "...",
        "answer_relevance": round(answer_relevance(q["question"], result["answer"]), 2),
        "faithfulness": round(faithfulness(result["answer"], result["sources"]), 2),
        "citation_correctness": round(citation_correctness(result["answer"], result["sources"]), 2),
        "hallucination_rate": round(hallucination_rate(result["answer"], result["sources"]), 2),
        "latency_s": round(result["latency_s"], 1),
    })
df = pd.DataFrame(rows)
df

,question,answer_relevance,faithfulness,citation_correctness,hallucination_rate,latency_s
0,What is the dimensionality of the model and th...,0.71,0.50,1.0,0.50,2.3
1,How many attention heads does the base Transfo...,0.83,0.75,1.0,0.25,1.8
2,What positional encoding does the Transformer ...,1.00,1.00,1.0,0.00,2.0
3,What BLEU score did the base Transformer achie...,1.00,1.00,1.0,0.00,1.8
4,Why is scaled dot-product attention scaled by ...,0.75,1.00,1.0,0.00,2.8


## 9.4 — DeepEval for LLM-judged groundedness

[DeepEval](https://github.com/confident-ai/deepeval) computes:

- **faithfulness** — every claim in the answer is supported by the retrieved context.
- **answer relevancy** — the answer addresses the question.
- **contextual precision** — retrieved chunks are ranked usefully.

It uses an LLM as judge. We point it at a local Ollama model (`gemma3:12b`) so everything stays free and offline.

> We use `gemma3:12b` as judge and `llama3.2:latest` as the generator — never judge with the same model that generated the answer, as it will be biased toward its own outputs.

In [15]:
import pandas as pd
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric, ContextualPrecisionMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import OllamaModel

# Separate generator and judge — never judge with the model that generated
answer_llm = LLMClient(model="llama3.2:latest")
judge = OllamaModel(model="gemma3:latest")

metrics = [
    FaithfulnessMetric(model=judge, threshold=0.7),
    AnswerRelevancyMetric(model=judge, threshold=0.7),
    ContextualPrecisionMetric(model=judge, threshold=0.7),
]

test_cases = []
for q in benchmark:
    res = rag_query(
        question=q["question"],
        vector_store=store,
        llm=answer_llm,
        top_k=5,
    )
    contexts = [
        source.text if hasattr(source, "text") else source["text"]
        for source in res["sources"]
    ]
    test_cases.append(
        LLMTestCase(
            input=q["question"],
            actual_output=res["answer"],
            retrieval_context=contexts,
            expected_output=q.get("ground_truth", ""),
        )
    )

# async_mode=False works around a Python 3.14 + anyio CancelScope incompatibility
results = evaluate(test_cases, metrics, async_config=AsyncConfig(run_async=False))
print(results)

✨ You're running DeepEval's latest Faithfulness Metric! (using gemma3:latest (Ollama), strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gemma3:latest (Ollama), strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using gemma3:latest (Ollama), strict=False, 
async_mode=False)...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:            How many attention heads does the base Transformer use?                                │
│  │     Actual Output:    The base Transformer uses 8 parallel attention layers, or heads. This is stated in     │
│  │                       section 3.2.3 of the provided document, where it is mentioned that "For each of        │
│  │                       these we use dk = dv = dmodel/h = 64" and that "h = 8 parallel attention layers".      │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Faithfulness         │ 0.33  │ 0.70      │ The score is 0.33 because while the context states    │
│              │                      │       │           │ that h = 8 and dk = dv = 64, the actual output        │
│              │                      │       │           │ contradicts this by stating that h = 8 parallel       │
│              │                      │       │           │ attention layers and dk = dv = dmodel/h = 64. This    │
│              │                      │       │           │ discrepancy suggests a low faithfulness score.        │
│        FAIL  │ Answer Relevancy     │ 0.67  │ 0.70      │ The score is 0.67 because the output includes a       │
│              │                      │       │           │ reference to a section of the document, but fails     │
│              │                      │       │           │ to directly answer the question about the number of   │
│              │                      │       │           │ attention heads in the base Transformer. It doesn't   │
│              │                      │       │           │ provide the specific numerical information            │
│              │                      │       │           │ requested.                                            │
│        PASS  │ Contextual Precision │ 1.00  │ 0.70      │ The score is 1.00 because the first retrieval c...    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_2                                                                                                 │
│  ├──   Input:            What positional encoding does the Transformer use and why?                             │
│  │     Actual Output:    The Transformer uses sinusoidal p

⚠ WARNING: No hyperparameters logged.
» ]8;id=5148928;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 107.57s | token cost: None)
» Test Results (5 total tests):
   » Pass Rate: 20.0% | Passed: 1 | Failed: 4

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Faithfulness', threshold=0.7, success=True, score=1.0, reason='The score is 1.00 because there are no contradictions identified, indicating perfect faithfulness between the actual output and the retrieval context.', strict_mode=False, evaluation_model='gemma3:latest (Ollama)', error=None, evaluation_cost=0.0, input_tokens=0, output_tokens=0, verbose_logs='Truths (limit=None):\n[\n    "The dimensionality of the input and output is dmodel = 512.",\n    "The inner-layer has dimensionality dff = 2048.",\n    "Learned embeddings are used to convert input and output tokens to vectors of dimension dmodel.",\n    "A learned linear transformation and softmax function are used to convert the decoder output to predicted next-token probabilities.",\n    "The model shares the same weight matrix between the two embedding layers and the pre-softmax linear transformation.",\n    "Each encoder layer has two sub-la

## 9.5 — Structured logging

Every production run goes through one logger. The format is JSONL: one event per line, machine-parseable, append-only.


In [16]:
from mrta.observability.logging import StructuredLogger
from mrta.core.config import settings
from pathlib import Path

logger = StructuredLogger()
LOG_FILE = Path(settings.log_file)

for q in benchmark:
    result = rag_query(q["question"], store, llm)
    logger.log_run(
        question=q["question"],
        answer=result["answer"],
        sources=result["sources"],
        latency_s=result["latency_s"],
    )

print(f"Logged {len(benchmark)} runs → {LOG_FILE}")

Logged 5 runs → data/logs/runs.jsonl


## 9.6 — Aggregating logs

A 5-line analyzer that turns the JSONL into a dashboard-ready DataFrame:


In [ ]:
import json
import pandas as pd
from pathlib import Path
from mrta.core.config import settings

LOG_FILE = Path(settings.log_file)

if not LOG_FILE.exists():
    print(f"No log file yet at {LOG_FILE} — run cell 9.5 first.")
else:
    events = [json.loads(line) for line in LOG_FILE.read_text().splitlines()]
    df_logs = pd.DataFrame(events)

    # Drop rows where sources is not a list (malformed from earlier runs)
    df_logs = df_logs[df_logs["sources"].apply(lambda x: isinstance(x, list))].copy()
    df_logs["n_sources"] = df_logs["sources"].apply(len)
    df_logs["question_short"] = df_logs["question"].str[:55] + "..."

    print(f"Total runs logged : {len(df_logs)}")
    print(f"Avg latency       : {df_logs['latency_s'].mean():.2f}s")
    print(f"p95 latency       : {df_logs['latency_s'].quantile(0.95):.2f}s")
    print(f"Avg sources / run : {df_logs['n_sources'].mean():.1f}")
    print()
    df_logs[["question_short", "latency_s", "n_sources"]].tail(5)

## 9.7 — Dockerfiles

Two Dockerfiles — one per process. Keeping them separate lets you scale the API independently of the UI, and means a Streamlit change doesn't require rebuilding the heavier API image.

Both use `python:3.11-slim` (matching CI and the local venv) and install only the extras each service needs — `[api]` for FastAPI, `[streamlit]` for the UI.

In [ ]:
from pathlib import Path

REPO_ROOT = Path(__file__).parent if "__file__" in dir() else Path.cwd()
while not (REPO_ROOT / "pyproject.toml").is_file() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

DOCKER_DIR = REPO_ROOT / "docker"

for fname in ["Dockerfile.api", "Dockerfile.streamlit"]:
    print(f"{'='*60}")
    print(f"  {fname}")
    print(f"{'='*60}")
    print((DOCKER_DIR / fname).read_text())
    print()

## 9.9 — The Qdrant swap

Once the vector store is Qdrant, the `VectorStore` class only changes inside its methods — the public API stays the same. That is the payoff for designing an interface in Phase 3 rather than calling FAISS directly from the RAG pipeline.

Sketch:

```python
from qdrant_client import QdrantClient, models

class QdrantStore:
    def __init__(self, url: str, dim: int, embedder, collection: str = "papers"):
        self.client = QdrantClient(url=url)
        if collection not in [c.name for c in self.client.get_collections().collections]:
            self.client.create_collection(
                collection_name=collection,
                vectors_config=models.VectorParams(size=dim, distance=models.Distance.COSINE),
            )
        self.collection = collection
        self.embedder = embedder

    def add(self, chunks):
        embs = self.embedder.encode([c.text for c in chunks], normalize_embeddings=True)
        self.client.upsert(self.collection, points=[
            models.PointStruct(id=i, vector=v.tolist(), payload=c.model_dump())
            for i, (c, v) in enumerate(zip(chunks, embs))
        ])

    def search(self, query: str, k: int = 5, doc_id: str | None = None):
        q = self.embedder.encode([query], normalize_embeddings=True)[0].tolist()
        flt = models.Filter(must=[models.FieldCondition(key="doc_id", match=models.MatchValue(value=doc_id))]) if doc_id else None
        res = self.client.search(self.collection, query_vector=q, query_filter=flt, limit=k)
        return [{**r.payload, "score": r.score} for r in res]
```

Now the `VECTOR_STORE` env var swaps backends with no callsite changes.


## 9.10 — CI on GitHub Actions

Drop this file at `.github/workflows/ci.yml`:


In [ ]:
# CI workflow already exists — see .github/workflows/ci.yml
# Runs on every push/PR to main: ruff → black --check → pytest

## 9.11 — README polish checklist

A senior-quality README hits all of these. Tick them off before announcing the repo:

- [ ] One-sentence elevator pitch at the top.
- [ ] A screenshot or animated GIF of the app working.
- [ ] An architecture diagram (Excalidraw or draw.io). Save as `docs/architecture.png`.
- [ ] A "Why these choices?" section — FAISS vs Qdrant, local vs API, chunk size.
- [ ] A reproducible **Quick start** that gets a stranger to a running demo in <5 minutes.
- [ ] An evaluation section with at least one table of metrics.
- [ ] A "Limitations" section that names what you would do next.
- [ ] A demo video link (5–8 min walkthrough on YouTube/Loom).

Honesty about limitations is the strongest senior signal — it shows you know where the project would break.


## What you learned

- A custom-metrics evaluation harness with `recall@k`, substring grounding, and citation-in-retrieval rate.
- How DeepEval plugs an LLM judge into the pipeline using a local Ollama model, keeping evaluation free and offline.
- Why to separate the generator model from the judge model — using the same model introduces self-serving bias.
- Structured JSONL logging with `structlog` for replayable runs.
- A two-Dockerfile + `docker-compose.yml` setup with Qdrant as the production vector store.
- The Qdrant swap sketch — payoff for designing a clean `VectorStore` interface earlier.
- A GitHub Actions CI workflow.
- A README polish checklist.

## Exercises

1. Expand the benchmark to 30 questions across 3 papers (e.g. AIAYN, BERT, GPT-2 / GPT-3).
2. Add a `latency-p95` SLO to CI: fail the build if it regresses above 10s.
3. Pipe the JSONL logs into a small Streamlit dashboard (`docs/dashboard.py`) that shows recall, latency, and cost over time.
4. Record a 5-minute demo video and link it from the README.

---

That's the full tutorial series. You now have:

1. A repo scaffold with `src/mrta/`, `apps/`, `docker/`, CI, and typed config.
2. Ten runnable tutorial notebooks (`notebooks/2026-05-25-phase00 ... phase09`).
3. An end-to-end working multimodal RAG assistant on local models.
4. The eval, observability, and deployment scaffolding to justify the "production-ready" label on your README.

Good luck — go ship it.